#### 발전소 구분 WS : 월성 KR : 고리 YK : 한빛 UJ : 한울
###### 월단위 원자력발전소 상태 : NuclearPlantStates
###### 분단위 실시간 폐수 수질현황: WasteWater
###### 10분단위 실시간 주변 방사선 량: RadioRate
###### 10분단위 실시간 온배수 현황: ThermalWasteWater
#### 본부 구분(2100:고리본부,2200:월성본부,2300:한빛본부,2400:한울본부,2800:새울본부)
###### 2024년 10월 이후 월단위 방사성 폐기물 발생량: RadioActiveWaste

In [1]:
%useLatestDescriptors
%use datetime
%use dataframe
%use ktor-client

In [2]:
val confFilePath = "/Users/unchil/Library/Application Support/Google/AndroidStudio2025.3.3/scratches/http-client.private.env.json"
val confDf = DataRow.readJson(path=confFilePath)

In [4]:

import kotlinx.datetime.format.byUnicodePattern
import kotlinx.datetime.format.FormatStringsInDatetimeFormats
import kotlinx.datetime.format.byUnicodePattern
import kotlin.time.Clock
@OptIn(kotlin.time.ExperimentalTime::class)
fun loadKHNP_Service(url:String): DataFrame<*> {
    val now = Clock.System.now()
    val genNames = listOf("WS", "KR", "YK", "SU", "UJ")
    val rows = mutableListOf<DataFrame<*>>()
    val myCollectionTime = now.toLocalDateTime(TimeZone.of("Asia/Seoul")).format(LocalDateTime.Format { byUnicodePattern("yyyy-MM-dd HH:mm") })

    genNames.forEach { genName ->
        val urlPath = url + "&genName=${genName}"
        try {
            val df_json = DataFrame.readJson( http.get(urlPath).deserializeJson().jsonString.byteInputStream())
            val instanceDf =
                df_json.get("response").get("body").get("items").get("item")[0] as DataFrame<*>

            val updatedDf = instanceDf.add {
                "collectionTime" from { myCollectionTime }
                "genName" from { genName }
            }
            rows.add(updatedDf)
        }catch(e:Exception){
            println(e.localizedMessage)
            println(urlPath)
        }
    }

    return rows.concat()
}

##### 분단위 실시간 폐수 수질현황: WasteWater

In [ ]:
val url = "${confDf.KHNP.host}/${confDf.KHNP.suburl[1]}?serviceKey=${confDf.KHNP.key}"
val df = loadKHNP_Service(url)

In [6]:
val updatedDf = df.update { name }.with {
    val currentName = it.toString() // 현재 행의 name 값
    when {
        currentName.contains("FLW00") || currentName.contains("TM001") -> "TM001"
        currentName.contains("PHY00") || currentName.contains("TM002") -> "TM002"
        else -> it // 조건에 해당하지 않으면 원래 값 유지
    }
}


In [8]:
val pivotedDf = updatedDf.pivot { name  }.groupBy { collectionTime and genName }.values { value and time }.flatten()

In [9]:
val renameDf = pivotedDf.rename(
    "value" to "tm001",
    "time" to "tm001_time",
    "value1" to "tm002",
    "time1" to "tm002_time"
)

###### 10분단위 실시간 온배수 현황: ThermalWasteWater

In [ ]:
val url3 = "${confDf.KHNP.host}/${confDf.KHNP.suburl[3]}?serviceKey=${confDf.KHNP.key}"
val df3 = loadKHNP_Service(url3)

RM001 : 취수구-수온 , RM002 : 취수구-염분, RM005 : 배수구-수온, RM006 : 배수구-염분

In [11]:
val updatedDf3 = df3.update { name }.with {
    val currentName = it.toString() // 현재 행의 name 값
    when {
        currentName.contains("RM001")-> "RM001"
        currentName.contains("RM002")-> "RM002"
        currentName.contains("RM005")-> "RM005"
        currentName.contains("RM006")-> "RM006"
        else -> it // 조건에 해당하지 않으면 원래 값 유지
    }
}

In [12]:
val pivotedDf3 = updatedDf3.pivot { name  }.groupBy { collectionTime and genName }.values { value and time }.flatten()

In [13]:
val renameDf3 = pivotedDf3.rename(
    "value" to "rm001",
    "time" to "rm001_time",
    "value1" to "rm002",
    "time1" to "rm002_time",
    "value2" to "rm005",
    "time2" to "rm005_time",
    "value3" to "rm006",
    "time3" to "rm006_time",
)